In [1]:
from pathlib import Path

import pandas as pd
import numpy as np
from PIL import Image

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from transformers import AutoImageProcessor, AutoModelForImageClassification
from tqdm.auto import tqdm
from sklearn.cluster import KMeans

#sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

DATA_DIR = Path("geo_dataset")  # change this
TRAIN_DIR = DATA_DIR / "train"
HOLDOUT_DIR = DATA_DIR / "holdout_public"
LABELS_PATH = DATA_DIR / "train_labels.csv"

/home/utn/poli22wo/miniconda3/envs/dl/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
OUTPUT_DIR = Path("outputs/cell_only_finetune")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

best_model_path = OUTPUT_DIR / "best_model.pt"
checkpoint_path = OUTPUT_DIR / "training_checkpoint.pt"
history_path = OUTPUT_DIR / "history.csv"

In [3]:
df = pd.read_csv(LABELS_PATH)

print(df.shape)
display(df.head())

(11758, 5)


,filename,country,iso,lat,lng
0,1fcb4a43864244259b7d8f4a00f1e475.jpg,Turkey,TR,40.112290,38.304629
1,742f45b0211c44ffb19ad84931ea519c.jpg,France,FR,48.094103,-1.994316
2,152a13ef249d4efa95c51ed93f026284.jpg,Turkey,TR,41.324741,27.961821
3,81ce4a88bff14fef8420bca42019b12b.jpg,France,FR,47.585855,-2.971004
4,6fbcfe523e1349759e6060d632d52e54.jpg,United_Kingdom,GB,55.698094,-4.305315


Validation Split

In [4]:
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["country"],
)

print("Training images:", len(train_df))
print("Validation images:", len(val_df))

Training images: 9406
Validation images: 2352


Creating a 64 Cell Grid over Europe

In [5]:
NUMBER_OF_CELLS = 64

kmeans = KMeans(
    n_clusters=NUMBER_OF_CELLS,
    random_state=42,
    n_init=10,
)

kmeans.fit(
    train_df[["lat", "lng"]]
)

train_df = train_df.copy()
val_df = val_df.copy()

train_df["cell_index"] = kmeans.labels_

val_df["cell_index"] = kmeans.predict(
    val_df[["lat", "lng"]]
)

cell_centres = (
    kmeans.cluster_centers_
    .astype(np.float32)
)

print("Cell centres:", cell_centres.shape)

Cell centres: (64, 2)


In [6]:
cell_counts = (
    train_df["cell_index"]
    .value_counts()
    .sort_index()
)

display(cell_counts)

print("Smallest cell:", cell_counts.min())
print("Largest cell:", cell_counts.max())

cell_index
0      70
1     151
2     157
3     184
4     205
     ... 
59    216
60    119
61    100
62    155
63     91
Name: count, Length: 64, dtype: int64

Smallest cell: 63
Largest cell: 289


Model Verification

In [8]:
MODEL_NAME = (
    "apple/mobilevitv2-1.0-imagenet1k-256"
)

processor = AutoImageProcessor.from_pretrained(
    MODEL_NAME
)

model = AutoModelForImageClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2 + NUMBER_OF_CELLS,
    ignore_mismatched_sizes=True,
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = model.to(device)

print("Device:", device)

[transformers] You passed `num_labels=66` which is incompatible to the `id2label` map of length `1000`.
Loading weights: 100%|██████████| 269/269 [00:00<00:00, 47542.04it/s]
[transformers] MobileViTV2ForImageClassification LOAD REPORT from: apple/mobilevitv2-1.0-imagenet1k-256
Key               | Status   |                                                                                           
------------------+----------+-------------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 512]) vs model:torch.Size([66, 512])
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([66])          

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


Device: cuda


In [9]:
total_params = sum(p.numel() for p in model.parameters())

print(f"Parameters: {total_params:,}")
assert total_params <= 5_000_000

Parameters: 4,422,699


In [10]:
geographic_model_path = Path(
    "outputs/geographic_cells/best_model.pt"
)

weights = torch.load(
    geographic_model_path,
    map_location=device,
    weights_only=True,
)

model.load_state_dict(weights)

<All keys matched successfully>

Normalizing grid cell centres

In [11]:
normalized_cell_centres = (
    cell_centres.copy()
)

normalized_cell_centres[:, 0] /= 90
normalized_cell_centres[:, 1] /= 180

In [12]:
cell_centres_tensor = torch.tensor(
    normalized_cell_centres,
    dtype=torch.float32,
    device=device,
)

In [13]:
print(cell_centres_tensor.shape)

torch.Size([64, 2])


Image Processor and Dataset

In [14]:
class GeolocationDataset(Dataset):
    def __init__(
        self,
        dataframe,
        image_dir,
        processor,
        transform=None,
    ):
        self.dataframe = dataframe.reset_index(drop=True)
        self.image_dir = Path(image_dir)
        self.processor = processor
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]

        image_path = self.image_dir / row["filename"]
        image = Image.open(image_path).convert("RGB")

        if self.transform is not None:
            image = self.transform(image)

        pixel_values = self.processor(
            images=image,
            return_tensors="pt",
        )["pixel_values"].squeeze(0)

        coordinates = torch.tensor(
            [
                row["lat"] / 90,
                row["lng"] / 180,
            ],
            dtype=torch.float32,
        )

        cell_index = torch.tensor(
            row["cell_index"],
            dtype=torch.long,
        )

        return pixel_values, coordinates, cell_index

In [15]:
train_dataset = GeolocationDataset(
    train_df,
    TRAIN_DIR,
    processor,
    transform=None,
)

val_dataset = GeolocationDataset(
    val_df,
    TRAIN_DIR,
    processor,
    transform=None,
)

In [16]:
BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

#Loss and Optimizer

In [17]:
coordinate_loss_function = nn.MSELoss()
cell_loss_function = nn.CrossEntropyLoss()

CELL_LOSS_WEIGHT = 0.01
COORDINATE_WEIGHT = 0.0

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-5,
)

In [18]:
images, coordinates, cell_labels = next(
    iter(train_loader)
)

images = images.to(device)
coordinates = coordinates.to(device)
cell_labels = cell_labels.to(device)

print("Images:", images.shape)
print("Coordinates:", coordinates.shape)
print("Cell labels:", cell_labels.shape)

Images: torch.Size([32, 3, 256, 256])
Coordinates: torch.Size([32, 2])
Cell labels: torch.Size([32])


In [19]:
def haversine_km(lat1, lng1, lat2, lng2):
    radius = 6371.0088

    lat1 = np.radians(lat1)
    lng1 = np.radians(lng1)
    lat2 = np.radians(lat2)
    lng2 = np.radians(lng2)

    difference = (
        np.sin((lat2 - lat1) / 2) ** 2
        + np.cos(lat1)
        * np.cos(lat2)
        * np.sin((lng2 - lng1) / 2) ** 2
    )

    return (
        2
        * radius
        * np.arcsin(
            np.sqrt(np.clip(difference, 0, 1))
        )
    )

In [20]:
torch.save(
    model.state_dict(),
    best_model_path,
)

Full Train + Validation Loop (Current best-> Epochs: 20, median: 709) (Reload Optimizer before continuing training)

In [21]:
start_epoch = 0
END_EPOCH = 15

history = []

best_median = 333.777435
best_epoch = 0

epochs_without_improvement = 0
patience = 4

for epoch in range(start_epoch, END_EPOCH):

    # --------------------
    # Training
    # --------------------
    model.train()
    total_training_loss = 0

    training_bar = tqdm(
        train_loader,
        desc=f"Epoch {epoch + 1}/{END_EPOCH} - Training",
    )

    for images, coordinates, cell_labels in training_bar:
        images = images.to(device)
        coordinates = coordinates.to(device)
        cell_labels = cell_labels.to(device)

        optimizer.zero_grad()

        outputs = model(
            pixel_values=images
        ).logits

        #Split the outputs
        direct_coordinates = torch.tanh(
            outputs[:, :2]
        )

        cell_logits = outputs[:, 2:]

        cell_probabilities = torch.softmax(
            cell_logits,
            dim=1,
        )

        cell_coordinates = (
            cell_probabilities
            @ cell_centres_tensor
        )

        final_coordinates = (
            COORDINATE_WEIGHT
            * direct_coordinates
            + (1 - COORDINATE_WEIGHT)
            * cell_coordinates
        )

        coordinate_loss = coordinate_loss_function(
            final_coordinates,
            coordinates,
        )

        cell_loss = cell_loss_function(
            cell_logits,
            cell_labels,
        )

        loss = (
            coordinate_loss
            + CELL_LOSS_WEIGHT * cell_loss
        )

        loss.backward()
        optimizer.step()

        total_training_loss += loss.item()

        training_bar.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    average_training_loss = (
        total_training_loss / len(train_loader)
    )

    # --------------------
    # Validation
    # --------------------
    model.eval()

    total_validation_loss = 0
    all_predictions = []
    all_coordinates = []

    validation_bar = tqdm(
        val_loader,
        desc=f"Epoch {epoch + 1}/{END_EPOCH} - Validation",
    )

    correct_cell_predictions = 0
    number_of_validation_images = 0

    with torch.no_grad():
        for images, coordinates, cell_labels in validation_bar:
            images = images.to(device)
            coordinates = coordinates.to(device)
            cell_labels = cell_labels.to(device)

            outputs = model(
                pixel_values=images
            ).logits

            # Direct coordinate prediction
            direct_coordinates = torch.tanh(
                outputs[:, :2]
            )

            # Cell prediction
            cell_logits = outputs[:, 2:]

            cell_probabilities = torch.softmax(
                cell_logits,
                dim=1,
            )

            # Probability-weighted cell coordinate
            cell_coordinates = (
                cell_probabilities
                @ cell_centres_tensor
            )

            # Final blended coordinate
            final_coordinates = (
                COORDINATE_WEIGHT
                * direct_coordinates
                + (1 - COORDINATE_WEIGHT)
                * cell_coordinates
            )

            # Evaluate the same blended coordinate used during training
            coordinate_loss = coordinate_loss_function(
                final_coordinates,
                coordinates,
            )

            cell_loss = cell_loss_function(
                cell_logits,
                cell_labels,
            )

            loss = (
                coordinate_loss
                + CELL_LOSS_WEIGHT * cell_loss
            )

            total_validation_loss += loss.item()

            # Store the final blended prediction, not the direct prediction
            all_predictions.append(
                final_coordinates.cpu().numpy()
            )

            all_coordinates.append(
                coordinates.cpu().numpy()
            )

            predicted_cells = cell_logits.argmax(
                dim=1
            )

            correct_cell_predictions += (
                predicted_cells == cell_labels
            ).sum().item()

            number_of_validation_images += (
                cell_labels.size(0)
            )     

    average_validation_loss = (
        total_validation_loss / len(val_loader)
    )

    cell_accuracy = (
        correct_cell_predictions
        / number_of_validation_images
    )


    # --------------------
    # Geographic metrics
    # --------------------
    all_predictions = np.concatenate(
        all_predictions
    )

    all_coordinates = np.concatenate(
        all_coordinates
    )

    predictions_degrees = all_predictions.copy()
    coordinates_degrees = all_coordinates.copy()

    predictions_degrees[:, 0] *= 90
    predictions_degrees[:, 1] *= 180

    coordinates_degrees[:, 0] *= 90
    coordinates_degrees[:, 1] *= 180

    distances = haversine_km(
        coordinates_degrees[:, 0],
        coordinates_degrees[:, 1],
        predictions_degrees[:, 0],
        predictions_degrees[:, 1],
    )

    mean_distance = np.mean(distances)
    median_distance = np.median(distances)
    within_200 = np.mean(distances < 200)
    within_750 = np.mean(distances < 750)

    history.append({
        "epoch": epoch + 1,
        "training_loss": average_training_loss,
        "validation_loss": average_validation_loss,
        "mean_km": mean_distance,
        "median_km": median_distance,
        "within_200": within_200,
        "within_750": within_750,
        "cell_accuracy": cell_accuracy,
    })

    print(f"\nEpoch {epoch + 1} results")
    print(f"Training loss: {average_training_loss:.4f}")
    print(f"Validation loss: {average_validation_loss:.4f}")
    print(f"Mean distance: {mean_distance:.1f} km")
    print(f"Median distance: {median_distance:.1f} km")
    print(f"Within 200 km: {within_200:.2%}")
    print(f"Within 750 km: {within_750:.2%}")
    print(f"Cell accuracy: {cell_accuracy:.2%}")

    # --------------------
    # Best model
    # --------------------
    if median_distance < best_median:
        best_median = median_distance
        best_epoch = epoch + 1

        epochs_without_improvement = 0

        torch.save(
            model.state_dict(),
            best_model_path,
        )

        print("Saved new best model.")

    else:
        epochs_without_improvement += 1

        print(
            "Epochs without improvement:",
            epochs_without_improvement,
        )

    # Save latest resumable checkpoint
    torch.save(
        {
            "epoch": epoch + 1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "best_median": best_median,
            "best_epoch": best_epoch,
            "history": history,
            "patience": patience,
            "epochs_without_improvement": (
                epochs_without_improvement
            ),
            "model_name": MODEL_NAME,
            "num_labels": 2 + NUMBER_OF_CELLS,
            "parameter_count": total_params,
            "number_of_cells": NUMBER_OF_CELLS,
            "cell_centres": (
                cell_centres_tensor
                .detach()
                .cpu()
            ),
            "coordinate_weight": COORDINATE_WEIGHT,
            "cell_loss_weight": CELL_LOSS_WEIGHT,
        },
        checkpoint_path,
    )

    # Preserve history after every epoch
    pd.DataFrame(history).to_csv(
        history_path,
        index=False,
    )

    if epochs_without_improvement >= patience:
        print("Early stopping.")
        break

Epoch 1/15 - Validation: 100%|██████████| 74/74 [00:12<00:00,  5.71it/s]



Epoch 1 results
Training loss: 0.0008
Validation loss: 0.0403
Mean distance: 584.0 km
Median distance: 334.5 km
Within 200 km: 37.41%
Within 750 km: 71.34%
Cell accuracy: 38.65%
Epochs without improvement: 1


Epoch 2/15 - Validation: 100%|██████████| 74/74 [00:14<00:00,  5.20it/s]



Epoch 2 results
Training loss: 0.0006
Validation loss: 0.0404
Mean distance: 575.0 km
Median distance: 320.9 km
Within 200 km: 37.20%
Within 750 km: 71.98%
Cell accuracy: 39.58%
Saved new best model.


Epoch 3/15 - Validation: 100%|██████████| 74/74 [00:12<00:00,  5.97it/s]


...


Epoch 4 results
Training loss: 0.0004
Validation loss: 0.0415
Mean distance: 580.2 km
Median distance: 326.6 km
Within 200 km: 37.29%
Within 750 km: 71.56%
Cell accuracy: 39.50%
Epochs without improvement: 2


Epoch 5/15 - Validation: 100%|██████████| 74/74 [00:11<00:00,  6.24it/s]



Epoch 5 results
Training loss: 0.0004
Validation loss: 0.0415
Mean distance: 576.3 km
Median distance: 324.1 km
Within 200 km: 37.37%
Within 750 km: 72.58%
Cell accuracy: 40.26%
Epochs without improvement: 3


Epoch 6/15 - Validation: 100%|██████████| 74/74 [00:12<00:00,  6.04it/s]



Epoch 6 results
Training loss: 0.0004
Validation loss: 0.0426
Mean distance: 574.8 km
Median distance: 329.5 km
Within 200 km: 37.29%
Within 750 km: 71.47%
Cell accuracy: 39.07%
Epochs without improvement: 4
Early stopping.


Diagnostic

In [22]:
best_weights = torch.load(
    best_model_path,
    map_location=device,
    weights_only=True,
)

model.load_state_dict(best_weights)
model.eval()

MobileViTV2ForImageClassification(
  (mobilevitv2): MobileViTV2Model(
    (conv_stem): MobileViTV2ConvLayer(
      (convolution): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (normalization): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (activation): SiLU()
    )
    (encoder): MobileViTV2Encoder(
      (layer): ModuleList(
        (0): MobileViTV2MobileNetLayer(
          (layer): ModuleList(
            (0): MobileViTV2InvertedResidual(
              (expand_1x1): MobileViTV2ConvLayer(
                (convolution): Conv2d(32, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
                (normalization): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
                (activation): SiLU()
              )
              (conv_3x3): MobileViTV2ConvLayer(
                (convolution): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), gr

In [23]:
all_cell_logits = []
all_true_coordinates = []

model.eval()

with torch.no_grad():
    for images, coordinates, cell_labels in tqdm(
        val_loader,
        desc="Collecting cell logits",
    ):
        images = images.to(device)

        outputs = model(
            pixel_values=images
        ).logits

        cell_logits = outputs[:, 2:]

        all_cell_logits.append(
            cell_logits.cpu()
        )

        all_true_coordinates.append(
            coordinates.numpy()
        )

In [24]:
all_cell_logits = torch.cat(
    all_cell_logits,
    dim=0,
)

all_true_coordinates = np.concatenate(
    all_true_coordinates
)

In [27]:
temperature_results = []

for temperature in [
    0.05,
    0.10,
    0.15,
    0.20,
    0.25,
    0.30,
    0.35,
    0.40,
]:
    cell_probabilities = torch.softmax(
        all_cell_logits / temperature,
        dim=1,
    ).numpy()

    predictions = (
        cell_probabilities
        @ normalized_cell_centres
    )

    predictions_degrees = predictions.copy()
    true_degrees = all_true_coordinates.copy()

    predictions_degrees[:, 0] *= 90
    predictions_degrees[:, 1] *= 180

    true_degrees[:, 0] *= 90
    true_degrees[:, 1] *= 180

    distances = haversine_km(
        true_degrees[:, 0],
        true_degrees[:, 1],
        predictions_degrees[:, 0],
        predictions_degrees[:, 1],
    )

    temperature_results.append({
        "temperature": temperature,
        "mean_km": np.mean(distances),
        "median_km": np.median(distances),
        "within_200": np.mean(distances < 200),
        "within_750": np.mean(distances < 750),
    })

In [28]:
display(
    pd.DataFrame(temperature_results)
)

,temperature,mean_km,median_km,within_200,within_750
0,0.05,616.268860,304.835266,0.415816,0.703656
1,0.10,613.114441,304.825745,0.412840,0.703231
2,0.15,610.039185,304.762451,0.413690,0.705357
3,0.20,606.854919,305.030548,0.416241,0.705782
4,0.25,603.727173,304.647217,0.416667,0.706207
5,0.30,600.790588,304.102905,0.411990,0.707483
6,0.35,597.974609,307.445801,0.408163,0.707908
7,0.40,595.284302,307.732361,0.406888,0.709184
